Note: Write your code in the code cells, and your responses in markdown. Run the entire script and display the outputs of your code. 

Due: **11:59PM Central Time on Friday, 10/24**.  Upload this notebook AND a HTML/PDF file to Canvas. Please make sure you have an AI disclaimer that indicates where/how you use AI tool. 

<!-- Helpful tips on Markdown: - https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Working%20With%20Markdown%20Cells.html# -->

In [ ]:
# Packages you might need: (pip install ... if you don't have them)
import pandas as pd # for data manipulation
import os  # for setting directory 
print(os.getcwd())
# os.chdir() # input your personal directory where the dataset is savedd
import statsmodels.formula.api as smf # for OLS regressions
from linearmodels.iv import IV2SLS # for IV2SLS
import numpy as np  # to work with arrays (vectors/matrices)

# Self Sufficiency Project 
- Background from Card and Hyslop (2005): 
    - In the Self Sufficiency Project (SSP), members of a randomly assigned treatmeng group could receive a subsidy for full-time work. The subsidy was available for 3 years, but only to people who began working full time within 12 months of random assignment.  
    - SSP provides two incentives: (1) find a job and leave welfare within a year, and (2) choose work over welfare in the longer term. 

- This question is designed to apply the IV-LATE framework to estimate the causal effects of SSP on leaving welfare, and characterize the compliers who would not have found a full-time job in the absence of the SSP subsidy. 

- The data set welfare.csv contains 5,480 observations for people in the SSP experiment. See PS4.pdf for variable definitions. 

In [ ]:
# Load dataset (using pandas "pd")
ssp = pd.read_csv("welfare.csv") # add your own directory if necessary
print(ssp.columns)
print(ssp['treatment'].value_counts(dropna=False)) 

In [ ]:
print(ssp['treatment'].value_counts()) # 1 if assigned to the treatment group (receiving a subsidy)
print(ssp['welfare15'].value_counts()) # 1 if on welfare at t=15. 

In [ ]:
# Note Missing Values: 
print(ssp[['ft15','ft20','ft24','ft48','treatment','welfare15','welfare20','welfare24','welfare48']].isnull().sum())
# input data in IV2SLS needs to be nonmissing in y/x1/x2/z 
# e.g., for t=24, input  data = ssp.loc[~ssp['ft15'].isnull()]

Note ft20/ft24/ft48 are missing in some rows. We need estimate the first-stage, the reduced form, and the 2SLS on the same sample, conditional on $FT_i(t)$ is nonmissing at a given $t$. For example, at $t=20$, estimate the regressions on the sample below: 

In [ ]:
t=20
print(ssp.loc[~ssp[f'ft{t}'].isnull()].head(2))  

## 1. First Stage
Estimate first stage models for the probability of working FT in months 15, 20, 24, 48, using treatment as the instrumental variable. $$FT_{i}(t)=\pi_{0}+\pi_{1}\text{treatment}_{i}+\epsilon_{i},\,\text{ for }t=15,20,24,28$$

In [ ]:
# Hint: model = smf.ols(f"welfare{t} ~ treatment", data=ssp.loc[~ssp[f'ft{t}'].isnull()]).fit(cov_type='HC1')

## 2. Reduced Form: Welfare recipience on random assignment. 
Estimate reduced form models for the probability of being on welfare in months 15, 20, 24, 48, using treatment as the instrumental variable. $$\text{Welfare}_{i}(t)=\delta_{0}+\delta_{1}\text{treatment}_{i}+\nu_{i},\,\text{ for }t=15,20,24,28$$ where $\text{Welfare}_{i}(t)=1$ if person $i$ is on welfare $t$ months since random assignment. 

In [ ]:
# Hint: make sure to estimate the regression conditional on "ft" is nonmissing at each given t; see the hint for first stage. 

## 3. Second Stage (2SLS) 
Estimate 2SLS models of the equation: $$\text{Welfare}_{i}(t)=\beta_{0}+\beta_{1}FT_{i}(t)+u_{i},\,\text{ for }t=15,20,24,28$$ using treatment as an instrument for $FT_i(t)$. Verify that the 2SLS estimate in each case is the ratio of the reduced form and first stage coefficients. Hint: using linearmodel IV2SLS. 

In [ ]:
# Example if using IV2SLS: 
# from linearmodels.iv import IV2SLS 
# model = IV2SLS.from_formula(
#     'y ~ 1 + x1 + [x2 ~ z1 + z2]',  # x2 is endogenous, z1,z2 are instruments
#     data=your_dataframe  # note the variables needs to nonmissing in the data
# )
# results = model.fit()
# print(results.summary)
# print(results.params)
print(ssp[['ft15','ft20','ft24','ft48','treatment','welfare15','welfare20','welfare24','welfare48']].isnull().sum())
# input data in IV2SLS needs to be nonmissing in y/x1/x2/z 
# e.g., for t=24, input  data = ssp.loc[~ssp['ft15'].isnull()]

In [ ]:
# Write your answer here: 

## 4. Characteristics of Compliers, Always Takers, and Never Takers
Find the mean characteristics of the compliers in month $t=15$: i.e., means of the variables imm, hsgrad, agelt25, age35p, working_at_baseline, anykidsu6, and nevermarried. Compare these to the characteristics of always takers and never takers in month 15. Hint: define outcome $Y_{i}=X_{i}\times FT_{i}(15)$. 

In [ ]:
print(ssp.columns)
X = ['imm', 'hsgrad', 'agelt25', 'age35p','working_at_baseline', 'anykidsu6', 'nevermarried']